# 0 · Ingest from Google Drive with Lakeflow Connect

Instead of downloading the sales CSVs from Kaggle and copying the contract PDFs
from this repo, this notebook ingests **both** from a Google Drive folder with the
managed **Lakeflow Connect Google Drive connector**:

```
<your Drive folder>/
├── structured/     dim_*.csv + fact_sales_normalized.csv  →  one Delta table per CSV
└── unstructured/   fictional_*.pdf contracts              →  files in the `contracts` UC volume
```

**What this notebook does**
1. Creates the target schema and the `contracts` volume (if missing)
2. Creates (or updates) **one serverless ingestion pipeline** with one object per CSV
   plus one object for the PDFs
3. Runs the pipeline and waits for it to finish
4. Verifies the row counts and the files that landed in the volume

When it finishes, the tables and the volume look exactly like the output of
`1_metric-views/0_IngestData` and `3_agent-bricks/0_ImportContractsToVolume`,
so you can **skip those two notebooks** and continue with `1_metric-views/1_CreateMetricView`.

> **Release status:** the Google Drive connector is in preview, and landing raw
> files into a volume (`format: FILE`) is **Beta**, so the pipeline runs on the
> `PREVIEW` channel. Check connector availability before using this stage on Databricks
> Free Edition.

### Before you start: prepare the Drive folder and the connection

1. **Drive folder.** Create a folder in Google Drive with two subfolders:
   - `structured/`: upload the CSVs from the
     [Kaggle retail star-schema dataset](https://www.kaggle.com/datasets/shrinivasv/retail-store-star-schema-dataset)
     (`dim_campaigns`, `dim_customers`, `dim_dates`, `dim_products`, `dim_salespersons`,
     `dim_stores`, `fact_sales_normalized`). `fact_sales_denormalized.csv` is not used.
   - `unstructured/`: upload the four PDFs from `3_agent-bricks/contracts/`.
2. **Unity Catalog connection.** In **Catalog Explorer → External data → Connections →
   Create connection**, pick **Google Drive**, sign in with the Google account that
   can read the folder, and note the connection name. OAuth sign-in is a one-time
   browser step and cannot be scripted. See
   [Create a Google Drive connection](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/google-drive-connection).
3. Copy the URLs of the two subfolders (`https://drive.google.com/drive/folders/<id>`)
   into the widgets below.

### Define Parameters

| Widget | Default | What it controls |
| --- | --- | --- |
| `CATALOG_NAME` | `workspace` | Unity Catalog for demo objects. |
| `SCHEMA_NAME` | `marketing_campaign` | Schema for the ingested tables and the contracts volume. |
| `CONNECTION_NAME` | *(empty)* | Name of the Google Drive UC connection from step 2. |
| `STRUCTURED_FOLDER_URL` | *(empty)* | URL of the Drive `structured/` subfolder (CSVs). |
| `UNSTRUCTURED_FOLDER_URL` | *(empty)* | URL of the Drive `unstructured/` subfolder (PDFs). |
| `VOLUME_NAME` | `contracts` | Volume where the PDFs land. Keep `contracts` so stage 3 finds them. |
| `PIPELINE_NAME` | `marketing_campaign_gdrive_ingestion` | Name of the Lakeflow Connect ingestion pipeline. |

In [ ]:
dbutils.widgets.text("CATALOG_NAME", "workspace", "CATALOG_NAME — Unity Catalog catalog for demo objects")
dbutils.widgets.text("SCHEMA_NAME", "marketing_campaign", "SCHEMA_NAME — schema inside the catalog")
dbutils.widgets.text("CONNECTION_NAME", "", "CONNECTION_NAME — Google Drive UC connection")
dbutils.widgets.text("STRUCTURED_FOLDER_URL", "", "STRUCTURED_FOLDER_URL — Drive folder with the CSVs")
dbutils.widgets.text("UNSTRUCTURED_FOLDER_URL", "", "UNSTRUCTURED_FOLDER_URL — Drive folder with the PDFs")
dbutils.widgets.text("VOLUME_NAME", "contracts", "VOLUME_NAME — UC volume for the contract PDFs")
dbutils.widgets.text("PIPELINE_NAME", "marketing_campaign_gdrive_ingestion", "PIPELINE_NAME — ingestion pipeline name")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
CONNECTION_NAME = dbutils.widgets.get("CONNECTION_NAME").strip()
STRUCTURED_FOLDER_URL = dbutils.widgets.get("STRUCTURED_FOLDER_URL").strip()
UNSTRUCTURED_FOLDER_URL = dbutils.widgets.get("UNSTRUCTURED_FOLDER_URL").strip()
VOLUME_NAME = dbutils.widgets.get("VOLUME_NAME")
PIPELINE_NAME = dbutils.widgets.get("PIPELINE_NAME")

for name, value in [("CONNECTION_NAME", CONNECTION_NAME),
                    ("STRUCTURED_FOLDER_URL", STRUCTURED_FOLDER_URL),
                    ("UNSTRUCTURED_FOLDER_URL", UNSTRUCTURED_FOLDER_URL)]:
    assert value, f"Set the {name} widget (see 'Before you start' above)."

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}/"
print("Target schema :", f"{CATALOG_NAME}.{SCHEMA_NAME}")
print("PDF volume    :", VOLUME_PATH)
print("Connection    :", CONNECTION_NAME)

### Create Schema and Volume

The volume must exist before the pipeline runs; the connector writes the PDFs into it.

In [ ]:
# Only create the catalog when it is missing: CREATE CATALOG IF NOT EXISTS still needs the
# metastore-level CREATE CATALOG privilege, even for a catalog that already exists.
existing_catalogs = {r[0] for r in spark.sql("SHOW CATALOGS").collect()}
if CATALOG_NAME not in existing_catalogs:
    spark.sql(f"CREATE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")
print("Schema and volume are ready.")

### Define the Ingestion Pipeline

One pipeline, one **object per destination table**:

- **Structured:** each CSV becomes its own Delta table. A `path_filter` selects the file
  and `schema_hints` pin the key, date, and amount columns to the same types the original
  `0_IngestData` notebook inferred, so the metric-view joins work unchanged.
- **Unstructured:** the PDFs are ingested with `format: FILE`. The file bytes are stored in
  the volume set by the `databricks.filespace-preview` table property (under generated ids,
  not the original names), and the `contract_files` table keeps one row per file with the
  original name in `_file_metadata.file_name` and the volume path in `file.uri`.
  `3_agent-bricks/1_CreateVectorSearchIndex` uses that mapping to keep contract names.

In [ ]:
SCHEMA_HINTS = {
    "dim_campaigns": "campaign_sk INT, start_date_sk INT, end_date_sk INT, campaign_budget INT",
    "dim_customers": "customer_sk INT",
    "dim_dates": "full_date DATE, date_sk INT, year INT, month INT, day INT, weekday INT, quarter INT",
    "dim_products": "product_sk INT",
    "dim_salespersons": "salesperson_sk INT",
    "dim_stores": "store_sk INT, store_manager_sk INT",
    "fact_sales_normalized": (
        "sales_sk INT, customer_sk INT, product_sk INT, store_sk INT, salesperson_sk INT, "
        "campaign_sk INT, sales_date TIMESTAMP, total_amount DOUBLE"
    ),
}

def csv_object(table, hints):
    return {"table": {
        "destination_catalog": CATALOG_NAME,
        "destination_schema": SCHEMA_NAME,
        "destination_table": table,
        "connector_options": {"gdrive_options": {
            "entity_type": "FILE",
            "url": STRUCTURED_FOLDER_URL,
            "file_ingestion_options": {
                "format": "CSV",
                "schema_evolution_mode": "NONE",
                "file_filters": [{"path_filter": f"*{table}.csv"}],
                "schema_hints": hints,
                "format_options": {"header": "true"},
            },
        }},
    }}

pdf_object = {"table": {
    "destination_catalog": CATALOG_NAME,
    "destination_schema": SCHEMA_NAME,
    "destination_table": "contract_files",
    "table_configuration": {"table_properties": {"databricks.filespace-preview": VOLUME_PATH}},
    "connector_options": {"gdrive_options": {
        "entity_type": "FILE",
        "url": UNSTRUCTURED_FOLDER_URL,
        "file_ingestion_options": {
            "format": "FILE",
            "schema_evolution_mode": "NONE",
            "file_filters": [{"path_filter": "*.pdf"}],
        },
    }},
}}

pipeline_spec = {
    "name": PIPELINE_NAME,
    "channel": "PREVIEW",          # required for format FILE (Beta)
    "catalog": CATALOG_NAME,
    "schema": SCHEMA_NAME,
    "ingestion_definition": {
        "connection_name": CONNECTION_NAME,
        "objects": [csv_object(t, h) for t, h in SCHEMA_HINTS.items()] + [pdf_object],
    },
}
print(f"{len(pipeline_spec['ingestion_definition']['objects'])} objects defined")

### Create (or Update) and Run the Pipeline

The pipeline is looked up by name, so re-running this notebook updates the existing
pipeline instead of creating a duplicate. Lakeflow Connect pipelines are triggered:
this cell starts one update and waits for it. To keep Drive in sync, schedule the
pipeline with a Lakeflow Job (`pipeline_task`).

In [ ]:
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

existing = [p for p in w.pipelines.list_pipelines(filter=f"name LIKE '{PIPELINE_NAME}'")
            if p.name == PIPELINE_NAME]
if existing:
    PIPELINE_ID = existing[0].pipeline_id
    w.api_client.do("PUT", f"/api/2.0/pipelines/{PIPELINE_ID}",
                    body={**pipeline_spec, "id": PIPELINE_ID})
    print("Updated pipeline", PIPELINE_ID)
else:
    PIPELINE_ID = w.api_client.do("POST", "/api/2.0/pipelines", body=pipeline_spec)["pipeline_id"]
    print("Created pipeline", PIPELINE_ID)

host = w.config.host.rstrip("/")
print(f"Pipeline UI: {host}/pipelines/{PIPELINE_ID}")

update_id = w.pipelines.start_update(pipeline_id=PIPELINE_ID).update_id
print("Started update", update_id, "(first run usually takes a few minutes)...")
while True:
    state = w.pipelines.get_update(pipeline_id=PIPELINE_ID, update_id=update_id).update.state.value
    if state in ("COMPLETED", "FAILED", "CANCELED"):
        break
    time.sleep(20)
print("Update finished with state:", state)
assert state == "COMPLETED", f"Pipeline update {state}. Open the pipeline UI above and check the event log."

### Verify

In [ ]:
for table in SCHEMA_HINTS:
    fqn = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table}"
    print(f"  {fqn}: {spark.table(fqn).count():,} rows")

# The connector stores each PDF in the volume under a generated id; the
# contract_files table maps it back to the original Drive file name.
display(spark.sql(f"""
  SELECT _file_metadata.file_name AS file_name, file.uri AS volume_path, size
  FROM {CATALOG_NAME}.{SCHEMA_NAME}.contract_files
  ORDER BY file_name
"""))